# SASRec авторов и разложение внешнего контекста

Ноутбук делает четыре вещи:

1. Обучает SASRec из репозитория авторов на MovieLens-1M. Эпоха выбирается по валидационным пользователям, а не по тестовым.
2. Повторяет контроль «лишние эпохи без контекста» на настоящем SASRec.
3. Выгружает состояния SASRec и прогоняет на них весь конвейер разложения: агрегации статьи, «похожие клиенты» против «контекста в целом», retrieval переходов и контроль g = 0.
4. По желанию запускает агрегации авторов в их собственном коде с `freeze=False`, с тем же числом эпох, что у контроля.

Время на бесплатной T4: шаги 1–3 занимают около 40 минут, шаг 4 ещё примерно 20 минут на каждую агрегацию.

**Перед запуском:** Runtime → Change runtime type → T4 GPU.

In [ ]:
!nvidia-smi -L
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 1. Код из архива

Загрузи `external_context_v3.zip`. Если сессия может оборваться, поставь `USE_DRIVE = True`: результаты будут сохраняться на Google Drive, и шаги можно будет продолжить с места остановки.

In [ ]:
import os, zipfile
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ext_work"
else:
    WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)

if not os.path.exists("/content/ext/code/run_context.py"):
    from google.colab import files
    up = files.upload()                      # выбери external_context_v3.zip
    zipfile.ZipFile(next(iter(up))).extractall("/content/ext")
print(sorted(os.listdir("/content/ext")), "WORK =", WORK)

## 2. Репозиторий авторов и данные

Пакет `recsys/src/data` выпал из репозитория авторов из-за правила `data/` в `.gitignore`. Здесь он восстанавливается из `torch_patch/recsys_src_data.py`. Данные берутся с сайта GroupLens. Должно получиться 1000209 строк.

In [ ]:
%cd /content
!test -d eca || git clone -q --depth 1 https://github.com/petrsokerin/External-Context-Aggregation.git eca
!mkdir -p eca/recsys/src/data eca/recsys/data/movielens_1m
!cp /content/ext/torch_patch/recsys_src_data.py eca/recsys/src/data/__init__.py
!cp /content/ext/torch_patch/sasrec_pipeline.py eca/recsys/
!test -f /content/ml-1m.zip || wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip -O /content/ml-1m.zip
!unzip -o -q /content/ml-1m.zip -d /content/
!cp /content/ml-1m/ratings.dat eca/recsys/data/movielens_1m/
!wc -l eca/recsys/data/movielens_1m/ratings.dat

## 3. Предобучение SASRec, выгрузка состояний, контроль лишних эпох

Гиперпараметры берутся из `config/main_movielens_1m.yaml` авторов: 150 эпох, AdamW, lr 0.001, max_len 200.

Тестовые пользователи авторов (20%) делятся на val (1/3) и test (2/3). Лучшая эпоха выбирается по val. После этого модель ещё 20 эпох учится без контекста с Adam, как второй этап авторов. Это контроль для шага 6.

Что должно получиться: строка `pretrained` с val и test в районе 0.07–0.09 (у авторов на их сплите 0.0776), затем 20 строк `no-context` и файл `sasrec_states.npz`. Около 15–25 минут.

In [ ]:
%cd /content/eca/recsys
!python sasrec_pipeline.py --config config/main_movielens_1m.yaml --work {WORK} --extra_epochs 20

## 4. Разложение на состояниях SASRec

Тот же numpy-конвейер, что в отчёте, но вместо MS-EMA он читает выгрузку SASRec. Идёт на CPU, около 8 минут.

- `run_experiments.py` повторяет агрегации статьи, раскладывает эффект на «похожих» и «контекст в целом» и проверяет шлюз.
- `run_shrink_control.py` делает контроль g = 0.
- `run_diagnostics.py` считает производные CE по доле контекста.

In [ ]:
os.environ["EXT_STATES"] = f"{WORK}/sasrec_states.npz"
os.environ["EXT_WORK"] = WORK
%cd /content/ext/code
!python run_context.py
!python run_experiments.py > {WORK}/experiments.stdout 2>&1; tail -n 5 {WORK}/experiments.stdout
!python run_shrink_control.py
!python run_diagnostics.py > {WORK}/diagnostics.log 2>&1; echo diagnostics done

## 5. Память переходов

Здесь причинный retrieval и все его варианты: похожие клиенты, похожие ситуации, окна по времени, утечка, случайный контроль, векторная форма и адаптивная λ. Около 10–12 минут на CPU.

In [ ]:
%cd /content/ext/code
!python run_retrieval.py > {WORK}/retrieval.stdout 2>&1; tail -n 3 {WORK}/retrieval.stdout
!python summarize.py

## 6. По желанию: агрегации авторов в их коде

Это `add_external_features` авторов с `freeze=False` и α = 0.1, 20 эпох, эпоха выбирается по val. Модель с контекстом сравнивается и с предобученной моделью, и с контролем «те же 20 эпох без контекста» из шага 3. Позиции раньше первого события пула получают пустой контекст: у авторов индекс −1 в `agg.py` указывает на будущее. Скрипт печатает, какая доля обучающих позиций и тестовых запросов попала бы под это в исходном коде.

Около 20 минут на каждую агрегацию. Если времени мало, оставь только `learnable_attention`.

In [ ]:
%cd /content/eca/recsys
!python sasrec_pipeline.py --config config/main_movielens_1m.yaml --work {WORK} --skip_pretrain --extra_epochs 0 --authors_agg learnable_attention,mean --freeze 0 --alpha 0.1 --ext_epochs 20
%cd /content/ext/code
!python summarize.py

## 7. Сохранить и прислать

Архив с JSON и логами без больших файлов. Пришли его мне вместе с выводом `summarize.py`.

In [ ]:
!cd {WORK} && zip -q -r /content/results_sasrec.zip . -x "*.npz" "*.pkl" "*.pth"
from google.colab import files
files.download("/content/results_sasrec.zip")

## Как читать результаты

- **Retrieval переходов значим, Mean и Learnable attention около нуля.** Тогда вывод с ML-1M переносится на трансформер. Если прирост retrieval меньше, чем на MS-EMA, это ожидаемо: сильной модели остаётся меньше, что исправлять.
- **Агрегация авторов выше предобученного SASRec, но не выше контроля лишних эпох.** Тогда прирост в статье объясняется протоколом, а не контекстом.
- **Контроль g = 0 даёт столько же, сколько G.** Тогда «контекст в целом» это калибровка, как на ритейле.
- **Производная по G положительна, по retrieval отрицательна.** Тогда диагностика до обучения работает и на SASRec.

**Если что-то пошло не так.**

- `ModuleNotFoundError: src.data`: не выполнилась ячейка шага 2, запусти её снова.
- CUDA out of memory на шаге 6: добавь `--eval_batch 16`.
- Сессия оборвалась: включи `USE_DRIVE = True` с самого начала. Тогда после переподключения выполни шаги 1 и 2 и продолжай с того шага, на котором остановился. Чекпойнт, выгрузка и контроль лишних эпох лежат в `WORK` на Drive, шаг 6 их подхватит через `--skip_pretrain`.
- Шаги 4 и 5 можно перезапускать по отдельности. Им нужен только `sasrec_states.npz` и кеш `cache_context.pkl` в `WORK`.